In [310]:
import pandas as pd
import numpy as np

**Trump 1.0 Data**

In [ ]:
#Using arrests because it can't be joined with the detention data on any unique identifier.
data2017 = pd.read_csv('data/arrests-2017.csv', header=4)
data2018 = pd.read_csv('data/arrests2018.csv', header=4)

/var/folders/1l/yh12s4qx29z26j7mg26bfby80000gn/T/ipykernel_23079/233313211.py:1: DtypeWarning: Columns (4,59,60,102) have mixed types. Specify dtype option on import or set low_memory=False.
  data2017 = pd.read_csv('data/arrests-2017.csv', header=4)
/var/folders/1l/yh12s4qx29z26j7mg26bfby80000gn/T/ipykernel_23079/233313211.py:2: DtypeWarning: Columns (4,58,59,60,102) have mixed types. Specify dtype option on import or set low_memory=False.
  data2018 = pd.read_csv('data/arrests2018.csv', header=4)


In [313]:
columns_to_keep = [
    "Apprehension Date And Time",
    "Apprehension AOR",
    "Most Serious Criminal Conviction Charge Code",
    "Most Serious Criminal Charge Status"
]

data2017 = data2017[columns_to_keep]
data2018 = data2018[columns_to_keep]

data2017 = pd.concat([data2017, data2018], ignore_index=True)

In [314]:
data2017['Apprehension Date And Time'] = pd.to_datetime(data2017['Apprehension Date And Time']).dt.strftime('%Y-%m-%d')
data2017 = data2017.rename(columns={'Apprehension Date And Time': 'Apprehension Date'})
data2017['Apprehension AOR'] = data2017['Apprehension AOR'].str.replace(' Area of Responsibility', '', regex=False)
data2017 = data2017.rename(columns={'Most Serious Criminal Charge Status': 'Criminality'})
data2017['Criminality'] = data2017['Criminality'].fillna('No Criminal Charges')
data2017['Criminality'] = data2017['Criminality'].replace('Turned Over to INS without Prosecution', 'No Criminal Charges')
data2017['Criminality'] = data2017['Criminality'].replace('Overturned', 'No Criminal Charges')
data2017['Criminality'] = data2017['Criminality'].replace('Dismissed', 'No Criminal Charges')
data2017['Criminality'] = data2017['Criminality'].replace('Pending', 'Pending Charges')


/var/folders/1l/yh12s4qx29z26j7mg26bfby80000gn/T/ipykernel_23079/3138188633.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data2017['Apprehension Date And Time'] = pd.to_datetime(data2017['Apprehension Date And Time']).dt.strftime('%Y-%m-%d')


In [ ]:
print("The number of missing AORs is",data2017['Apprehension AOR'].isna().sum())
data2017 = data2017.dropna(subset=['Apprehension AOR'])

1077
0


In [ ]:
#Getting scraped codebook
crimelist = pd.read_csv("data/crimeclass.csv")
df = crimelist.rename(columns={'Type of Offense Code  V=violent  D=drug-related  Blank = nonviolent or not drug related': 'Type'})
violentcrimes = df[df['Type'].str.contains('V', na=False)]

In [324]:

data2017 = data2017.merge(violentcrimes, left_on = "Most Serious Criminal Conviction Charge Code", right_on = "NCIC Offense Code", how = 'left')


In [325]:

data2017['Type'] = np.where(((data2017['Type'] != "V") & (data2017['Criminality'] == "Convicted")), "Convicted - Non-violent", data2017['Type'])
data2017['Type'] = np.where((data2017['Type'] == "V"), "Convicted - Violent crime", data2017['Type'])
data2017['Type'] = np.where(((data2017['Type'] != "V") & (data2017['Criminality'] == "Pending Charges")), "Pending Charges", data2017['Type'])

data2017

,Apprehension Date,Apprehension AOR,Most Serious Criminal Conviction Charge Code,Criminality,NCIC Offense Code,Description of Crime,Type
0,2017-08-16,Phoenix,1305,Convicted,1305,AGGRAV ASSLT - NONFAMILY-WEAPON,Convicted - Violent crime
1,2017-07-18,Phoenix,1312,Convicted,1312,AGGRAV ASSLT-- POL OFF-STGARM,Convicted - Violent crime
2,2016-11-20,Phoenix,1315,Convicted,1315,AGGRAV ASSLT - WEAPON,Convicted - Violent crime
3,2017-08-10,Phoenix,3560,Convicted,NaN,NaN,Convicted - Non-violent
4,2017-08-17,Phoenix,5015,Convicted,NaN,NaN,Convicted - Non-violent
...,...,...,...,...,...,...,...
291000,2018-02-27,Los Angeles,3803,Convicted,NaN,NaN,Convicted - Non-violent
291001,2018-08-30,Phoenix,5212,Convicted,NaN,NaN,Convicted - Non-violent
291002,2018-01-23,Buffalo,0301,Convicted,NaN,NaN,Convicted - Non-violent
291003,2018-06-28,Miami,3704,Pending Charges,NaN,NaN,Pending Charges


In [326]:

data2017['Type'] = data2017['Type'].fillna('No Criminal Charges')

In [327]:
data2017 = data2017.drop(columns=['Most Serious Criminal Conviction Charge Code', 'NCIC Offense Code', 'Description of Crime'])
data2017


,Apprehension Date,Apprehension AOR,Criminality,Type
0,2017-08-16,Phoenix,Convicted,Convicted - Violent crime
1,2017-07-18,Phoenix,Convicted,Convicted - Violent crime
2,2016-11-20,Phoenix,Convicted,Convicted - Violent crime
3,2017-08-10,Phoenix,Convicted,Convicted - Non-violent
4,2017-08-17,Phoenix,Convicted,Convicted - Non-violent
...,...,...,...,...
291000,2018-02-27,Los Angeles,Convicted,Convicted - Non-violent
291001,2018-08-30,Phoenix,Convicted,Convicted - Non-violent
291002,2018-01-23,Buffalo,Convicted,Convicted - Non-violent
291003,2018-06-28,Miami,Pending Charges,Pending Charges


In [328]:
data2017 = data2017[data2017['Apprehension Date'] <= '2017-10-15']
print(data2017['Apprehension Date'].max())
print(data2017['Apprehension Date'].min())

2017-10-15
2016-10-01


**Trump 2.0 Data**

In [329]:
data2025 = pd.read_csv('data/arrests2025clean.csv')

In [330]:
columns_to_keep3 = ["apprehension_date", "apprehension_aor", "apprehension_criminality", "unique_identifier"]
data2025 = data2025[columns_to_keep3]
data2025filtered = data2025[data2025['apprehension_date'] >= '2024-10-01'].copy()
data2025filtered
data2025filtered['unique_identifier'].nunique()

238306

In [331]:
detdata2025 = pd.read_csv('data/detentions2025.csv')
print(detdata2025.columns.tolist())

/var/folders/1l/yh12s4qx29z26j7mg26bfby80000gn/T/ipykernel_23079/3963119366.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  detdata2025 = pd.read_csv('data/detentions2025.csv')


['stay_ID', 'n_stints', 'detention_facility_codes_all', 'stay_book_in_date_time', 'stay_book_out_date_time', 'detention_release_reason', 'stay_book_out_date', 'stay_release_reason', 'religion', 'gender', 'marital_status', 'birth_year', 'ethnicity', 'entry_status', 'felon', 'bond_posted_date', 'bond_posted_amount', 'case_status', 'case_category', 'final_order_yes_no', 'final_order_date', 'case_threat_level', 'book_in_criminality', 'final_charge', 'departed_date', 'departure_country', 'initial_bond_set_amount', 'citizenship_country', 'final_program', 'most_serious_conviction_code', 'msc_charge', 'unique_identifier', 'n_stays', 'detention_facility_first', 'detention_facility_code_first', 'book_in_date_time_first', 'book_out_date_time_first', 'detention_facility_longest', 'detention_facility_code_longest', 'book_in_date_time_longest', 'book_out_date_time_longest', 'detention_facility_last', 'detention_facility_code_last', 'book_in_date_time_last', 'book_out_date_time_last']


In [332]:
detdata2025

,stay_ID,n_stints,detention_facility_codes_all,stay_book_in_date_time,stay_book_out_date_time,detention_release_reason,stay_book_out_date,stay_release_reason,religion,gender,...,book_in_date_time_first,book_out_date_time_first,detention_facility_longest,detention_facility_code_longest,book_in_date_time_longest,book_out_date_time_longest,detention_facility_last,detention_facility_code_last,book_in_date_time_last,book_out_date_time_last
0,1072f6b200bcdc1ce54adfdb963e816c237b0098_2025-...,1,ELZICDF,2025-10-16 00:28:00 UTC,NaN,NaN,NaN,NaN,NaN,Female,...,2025-10-16 00:28:00 UTC,NaN,ELIZABETH CONTRACT D.F.,ELZICDF,2025-10-16 00:28:00 UTC,NaN,ELIZABETH CONTRACT D.F.,ELZICDF,2025-10-16 00:28:00 UTC,NaN
1,aa371276d11cf1284d107f007f62f8de9b8c47f7_2025-...,1,EROFCB,2025-10-15 23:52:00 UTC,NaN,NaN,NaN,NaN,NaN,Male,...,2025-10-15 23:52:00 UTC,NaN,ERO EL PASO CAMP EAST MONTANA,EROFCB,2025-10-15 23:52:00 UTC,NaN,ERO EL PASO CAMP EAST MONTANA,EROFCB,2025-10-15 23:52:00 UTC,NaN
2,1aca11d84c476ce0d616931f4f79acf65ca65ff0_2025-...,2,CSDHOLD; DENICDF,2025-10-15 23:51:00 UTC,NaN,NaN,NaN,NaN,NaN,Male,...,2025-10-15 23:51:00 UTC,2025-10-15 23:54:00 UTC,DENVER CONTRACT DETENTION FACILITY,DENICDF,2025-10-15 23:55:00 UTC,NaN,DENVER CONTRACT DETENTION FACILITY,DENICDF,2025-10-15 23:55:00 UTC,NaN
3,8a8a3ac2ae41aa4cdadc988dac43af0824be0254_2025-...,1,MTGHOLD,2025-10-15 23:39:00 UTC,NaN,NaN,NaN,NaN,NaN,Female,...,2025-10-15 23:39:00 UTC,NaN,MONTGOMERY HOLD RM,MTGHOLD,2025-10-15 23:39:00 UTC,NaN,MONTGOMERY HOLD RM,MTGHOLD,2025-10-15 23:39:00 UTC,NaN
4,0692fe0a0be09640ee7b14f352b0462ee4aa5e0e_2025-...,1,ELVDFTX,2025-10-15 23:34:00 UTC,NaN,NaN,NaN,NaN,NaN,Male,...,2025-10-15 23:34:00 UTC,NaN,EL VALLE DETENTION FACILITY,ELVDFTX,2025-10-15 23:34:00 UTC,NaN,EL VALLE DETENTION FACILITY,ELVDFTX,2025-10-15 23:34:00 UTC,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
671745,233164440aaf5b33529f5fc064efb9849eef9473_2014-...,2,FSF; PIC,2014-04-09 18:20:00 UTC,2025-08-08 04:57:00 UTC,Removed,2025-08-08,Removed,NaN,Male,...,2014-04-09 18:20:00 UTC,2025-08-05 12:22:00 UTC,FLORENCE STAGING FACILITY,FSF,2014-04-09 18:20:00 UTC,2025-08-05 12:22:00 UTC,PORT ISABEL SPC,PIC,2025-08-05 12:23:00 UTC,2025-08-08 04:57:00 UTC
671746,bf8cd17531ceeb9de84364fe797c9d978436c3c0_2014-...,1,FSF,2014-02-25 19:43:00 UTC,2024-02-13 07:36:00 UTC,Bonded Out - IJ,2024-02-13,Bonded Out - IJ,NaN,Male,...,2014-04-25 22:25:00 UTC,2024-02-13 07:36:00 UTC,FLORENCE STAGING FACILITY,FSF,2014-04-25 22:25:00 UTC,2024-02-13 07:36:00 UTC,FLORENCE STAGING FACILITY,FSF,2014-04-25 22:25:00 UTC,2024-02-13 07:36:00 UTC
671747,c637e9dd42ba599b9f6bd3f9aa142c4bb9192b2d_2014-...,2,FREHOLD; LOSHOLD,2014-01-17 13:18:00 UTC,2025-02-08 05:28:00 UTC,Removed,2025-02-08,Removed,NaN,Male,...,2014-01-17 13:18:00 UTC,2025-02-07 16:00:00 UTC,FRESNO HOLDROOM,FREHOLD,2014-01-17 13:18:00 UTC,2025-02-07 16:00:00 UTC,LOS CUST CASE,LOSHOLD,2025-02-08 02:00:00 UTC,2025-02-08 05:28:00 UTC
671748,a74918dcaad3e2f00550f861202538c60c07161c_2011-...,3,YORCOPA; MIAHOLD; FLDSSFS,2011-01-11 23:59:00 UTC,NaN,NaN,NaN,NaN,NaN,Male,...,2011-01-13 19:27:00 UTC,2025-10-10 15:49:00 UTC,"YORK COUNTY JAIL, PA",YORCOPA,2011-01-13 19:27:00 UTC,2025-10-10 15:49:00 UTC,FLORIDA SOFT-SIDED FACILITY-SOUTH,FLDSSFS,2025-10-11 23:00:00 UTC,NaN


In [ ]:
columns_to_keep2 = [
    "stay_book_in_date_time",
    "most_serious_conviction_code",
    "msc_charge",
    "unique_identifier"
]

detdata2025 = detdata2025[columns_to_keep2]
detdatafiltered = detdata2025[detdata2025['stay_book_in_date_time'] >= '2024-10-01'].copy()

detdatafiltered['unique_identifier'].nunique()


326659

In [334]:
detdatafiltered

,stay_book_in_date_time,most_serious_conviction_code,msc_charge,unique_identifier
0,2025-10-16 00:28:00 UTC,NaN,NaN,1072f6b200bcdc1ce54adfdb963e816c237b0098
1,2025-10-15 23:52:00 UTC,NaN,NaN,aa371276d11cf1284d107f007f62f8de9b8c47f7
2,2025-10-15 23:51:00 UTC,NaN,NaN,1aca11d84c476ce0d616931f4f79acf65ca65ff0
3,2025-10-15 23:39:00 UTC,NaN,NaN,8a8a3ac2ae41aa4cdadc988dac43af0824be0254
4,2025-10-15 23:34:00 UTC,NaN,NaN,0692fe0a0be09640ee7b14f352b0462ee4aa5e0e
...,...,...,...,...
337722,2024-10-01 00:38:00 UTC,5404,Driving Under Influence Liquor,8b86349cdab4b7ef992f81e94ae98ce3491392aa
337723,2024-10-01 00:30:00 UTC,NaN,NaN,06ae64f41bcc1ab0f98de3e5dceede20df54a4c7
337724,2024-10-01 00:30:00 UTC,NaN,NaN,4ae498326dc0730cd34e85f18995eb0d63e868a9
337725,2024-10-01 00:30:00 UTC,NaN,NaN,d890ad6b639b98ac00ac1751802101caca64d2cb


In [ ]:
mergeddata2025 = data2025.merge(detdata2025, on = "unique_identifier", how = 'right')
mergeddata2025['apprehension_date'] = pd.to_datetime(mergeddata2025['apprehension_date'])
mergeddata2025 = mergeddata2025[mergeddata2025['apprehension_date'] >= '2024-10-01']

In [ ]:
#Checking the merge
print(mergeddata2025['apprehension_aor'].isna().sum())
mergeddata2025 = mergeddata2025.dropna(subset=['apprehension_aor'])
print(mergeddata2025['stay_book_in_date_time'].isna().sum())
print(detdata2025['unique_identifier'].isna().sum())

338227
0


In [ ]:
mergeddata2025 = mergeddata2025.replace(to_replace="1 Convicted Criminal", value="Convicted")
mergeddata2025 = mergeddata2025.replace(to_replace="2 Pending Criminal Charges", value="Pending Charges")
mergeddata2025 = mergeddata2025.replace(to_replace="3 Other Immigration Violator", value="No Criminal Charges")

In [ ]:
mergeddata2025[mergeddata2025.duplicated(subset=['unique_identifier', 'apprehension_date', 'apprehension_aor'], keep=False)] #52490 dupplicates

,apprehension_date,apprehension_aor,apprehension_criminality,unique_identifier,stay_book_in_date_time,most_serious_conviction_code,msc_charge
1,2025-10-15,El Paso Area of Responsibility,2 Pending Criminal Charges,aa371276d11cf1284d107f007f62f8de9b8c47f7,2025-10-15 23:52:00 UTC,NaN,NaN
2,2025-06-11,New Orleans Area of Responsibility,2 Pending Criminal Charges,aa371276d11cf1284d107f007f62f8de9b8c47f7,2025-10-15 23:52:00 UTC,NaN,NaN
4,2025-10-15,Houston Area of Responsibility,2 Pending Criminal Charges,8a8a3ac2ae41aa4cdadc988dac43af0824be0254,2025-10-15 23:39:00 UTC,NaN,NaN
5,2025-07-17,Houston Area of Responsibility,2 Pending Criminal Charges,8a8a3ac2ae41aa4cdadc988dac43af0824be0254,2025-10-15 23:39:00 UTC,NaN,NaN
12,2025-04-06,New Orleans Area of Responsibility,2 Pending Criminal Charges,7566da657aae170b44a2bd9025662c85c63a0e25,2025-10-15 23:28:00 UTC,NaN,NaN
...,...,...,...,...,...,...,...
706271,2025-06-16,New York City Area of Responsibility,1 Convicted Criminal,09f6e84c91aa528fabf7381175cde781ca26f306,2019-03-28 09:22:00 UTC,1399,Assault
706278,2025-06-17,Los Angeles Area of Responsibility,1 Convicted Criminal,21f681919f1d97fffada2c71794ed61056f059ab,2018-10-11 10:27:00 UTC,0999,Homicide
706279,2025-06-16,Los Angeles Area of Responsibility,1 Convicted Criminal,21f681919f1d97fffada2c71794ed61056f059ab,2018-10-11 10:27:00 UTC,0999,Homicide
706286,2025-06-18,Denver Area of Responsibility,1 Convicted Criminal,8f9f48769b12fb9d3a55c032982b3e5009561681,2017-05-25 11:53:00 UTC,3560,Marijuana - Sell


In [ ]:
#Dropping duplicates
mergeddata2025.drop_duplicates(subset=['unique_identifier', 'apprehension_date', 'apprehension_aor'], inplace=True)

/var/folders/1l/yh12s4qx29z26j7mg26bfby80000gn/T/ipykernel_23079/471240534.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mergeddata2025.drop_duplicates(subset=['unique_identifier', 'apprehension_date', 'apprehension_aor'], inplace=True)


In [ ]:
mergeddata2025 = mergeddata2025.rename(columns={'apprehension_date': 'Apprehension Date'})
mergeddata2025 = mergeddata2025.rename(columns={'apprehension_aor': 'Apprehension AOR'})
mergeddata2025 = mergeddata2025.rename(columns={'apprehension_criminality': 'Criminality'})

In [ ]:
mergeddata2025= mergeddata2025.merge(violentcrimes, left_on = "most_serious_conviction_code", right_on = "NCIC Offense Code", how = 'left')

,Apprehension Date,Apprehension AOR,Criminality,unique_identifier,stay_book_in_date_time,most_serious_conviction_code,msc_charge,NCIC Offense Code,Description of Crime,Type
0,2025-10-15,Newark Area of Responsibility,2 Pending Criminal Charges,1072f6b200bcdc1ce54adfdb963e816c237b0098,2025-10-16 00:28:00 UTC,NaN,NaN,NaN,NaN,NaN
1,2025-10-15,El Paso Area of Responsibility,2 Pending Criminal Charges,aa371276d11cf1284d107f007f62f8de9b8c47f7,2025-10-15 23:52:00 UTC,NaN,NaN,NaN,NaN,NaN
2,2025-06-11,New Orleans Area of Responsibility,2 Pending Criminal Charges,aa371276d11cf1284d107f007f62f8de9b8c47f7,2025-10-15 23:52:00 UTC,NaN,NaN,NaN,NaN,NaN
3,2025-10-15,Denver Area of Responsibility,2 Pending Criminal Charges,1aca11d84c476ce0d616931f4f79acf65ca65ff0,2025-10-15 23:51:00 UTC,NaN,NaN,NaN,NaN,NaN
4,2025-10-15,Houston Area of Responsibility,2 Pending Criminal Charges,8a8a3ac2ae41aa4cdadc988dac43af0824be0254,2025-10-15 23:39:00 UTC,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
230852,2025-01-27,San Diego Area of Responsibility,3 Other Immigration Violator,7cb498fa5244cfb3a2d5b79772fe3993bac292cd,2018-06-08 07:42:00 UTC,NaN,NaN,NaN,NaN,NaN
230853,2025-06-18,Denver Area of Responsibility,1 Convicted Criminal,8f9f48769b12fb9d3a55c032982b3e5009561681,2017-05-25 11:53:00 UTC,3560,Marijuana - Sell,NaN,NaN,NaN
230854,2025-02-07,San Francisco Area of Responsibility,1 Convicted Criminal,c637e9dd42ba599b9f6bd3f9aa142c4bb9192b2d,2014-01-17 13:18:00 UTC,5403,Driving Under Influence Drugs,NaN,NaN,NaN
230855,2025-10-10,Miami Area of Responsibility,2 Pending Criminal Charges,a74918dcaad3e2f00550f861202538c60c07161c,2011-01-11 23:59:00 UTC,NaN,NaN,NaN,NaN,NaN


In [351]:
mergeddata2025['Type'].unique()
mergeddata2025['Type'].value_counts()

Type
V    16213
Name: count, dtype: int64

In [ ]:
mergeddata2025['Type'] = np.where(((mergeddata2025['Type'] != "V") & (mergeddata2025['Criminality'] == "Convicted")), "Convicted - Non-violent", mergeddata2025['Type'])
mergeddata2025['Type'] = np.where((mergeddata2025['Type'] == "V"), "Convicted - Violent crime", mergeddata2025['Type'])
mergeddata2025['Type'] = np.where(((mergeddata2025['Type'] != "V") & (mergeddata2025['Criminality'] == "Pending Charges")), "Pending Charges", mergeddata2025['Type'])
mergeddata2025['Type'] = mergeddata2025['Type'].fillna('No Criminal Charges')

,Apprehension Date,Apprehension AOR,Criminality,unique_identifier,stay_book_in_date_time,most_serious_conviction_code,msc_charge,NCIC Offense Code,Description of Crime,Type
0,2025-10-15,Newark Area of Responsibility,2 Pending Criminal Charges,1072f6b200bcdc1ce54adfdb963e816c237b0098,2025-10-16 00:28:00 UTC,NaN,NaN,NaN,NaN,Pending Charges
1,2025-10-15,El Paso Area of Responsibility,2 Pending Criminal Charges,aa371276d11cf1284d107f007f62f8de9b8c47f7,2025-10-15 23:52:00 UTC,NaN,NaN,NaN,NaN,Pending Charges
2,2025-06-11,New Orleans Area of Responsibility,2 Pending Criminal Charges,aa371276d11cf1284d107f007f62f8de9b8c47f7,2025-10-15 23:52:00 UTC,NaN,NaN,NaN,NaN,Pending Charges
3,2025-10-15,Denver Area of Responsibility,2 Pending Criminal Charges,1aca11d84c476ce0d616931f4f79acf65ca65ff0,2025-10-15 23:51:00 UTC,NaN,NaN,NaN,NaN,Pending Charges
4,2025-10-15,Houston Area of Responsibility,2 Pending Criminal Charges,8a8a3ac2ae41aa4cdadc988dac43af0824be0254,2025-10-15 23:39:00 UTC,NaN,NaN,NaN,NaN,Pending Charges
...,...,...,...,...,...,...,...,...,...,...
230852,2025-01-27,San Diego Area of Responsibility,3 Other Immigration Violator,7cb498fa5244cfb3a2d5b79772fe3993bac292cd,2018-06-08 07:42:00 UTC,NaN,NaN,NaN,NaN,NaN
230853,2025-06-18,Denver Area of Responsibility,1 Convicted Criminal,8f9f48769b12fb9d3a55c032982b3e5009561681,2017-05-25 11:53:00 UTC,3560,Marijuana - Sell,NaN,NaN,Convicted - Non-violent
230854,2025-02-07,San Francisco Area of Responsibility,1 Convicted Criminal,c637e9dd42ba599b9f6bd3f9aa142c4bb9192b2d,2014-01-17 13:18:00 UTC,5403,Driving Under Influence Drugs,NaN,NaN,Convicted - Non-violent
230855,2025-10-10,Miami Area of Responsibility,2 Pending Criminal Charges,a74918dcaad3e2f00550f861202538c60c07161c,2011-01-11 23:59:00 UTC,NaN,NaN,NaN,NaN,Pending Charges


In [353]:
mergeddata2025['Type'] = mergeddata2025['Type'].fillna('No Criminal Charges')

In [354]:
mergeddata2025['Type'].unique()
mergeddata2025['Type'].value_counts()

Type
Convicted - Non-violent      78464
Pending Charges              71483
No Criminal Charges          64791
Convicted - Violent crime    16119
Name: count, dtype: int64

In [355]:
mergeddata2025['Apprehension AOR'] = mergeddata2025['Apprehension AOR'].str.replace(' Area of Responsibility', '', regex=False)

In [356]:
mergeddata2025 = mergeddata2025.drop(columns=['most_serious_conviction_code', 'NCIC Offense Code', 'Description of Crime', 'unique_identifier', 'stay_book_in_date_time', 'msc_charge'])

In [357]:
print(mergeddata2025['Apprehension Date'].max())
print(mergeddata2025['Apprehension Date'].min())


2025-10-16 00:00:00
2024-10-01 00:00:00


In [358]:
mergeddata2025 = mergeddata2025.sort_values(by=["Type"])

In [ ]:
mergeddata2025['Type'].unique()
mergeddata2025['Type'].value_counts()

Type
Convicted - Non-violent      78464
Pending Charges              71483
No Criminal Charges          64791
Convicted - Violent crime    16119
Name: count, dtype: int64

***Combining the Datasets***

In [361]:
mergeddata2025['Administration'] = 'Trump 2'
data2017['Administration'] = 'Trump 1'
combined_data = pd.concat([mergeddata2025, data2017], ignore_index=True)
combined_data['Apprehension Date'] = pd.to_datetime(combined_data['Apprehension Date']).dt.strftime('%Y-%m-%d')

combined_data['Type'].unique()
combined_data['Type'].value_counts()

/var/folders/1l/yh12s4qx29z26j7mg26bfby80000gn/T/ipykernel_23079/2273868696.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2017['Administration'] = 'Trump 1'


Type
Convicted - Non-violent      157747
Pending Charges              103242
No Criminal Charges           80861
Convicted - Violent crime     33665
Name: count, dtype: int64

In [362]:
combined_data

,Apprehension Date,Apprehension AOR,Criminality,Type,Administration
0,2025-08-18,Chicago,1 Convicted Criminal,Convicted - Non-violent,Trump 2
1,2025-04-04,Seattle,1 Convicted Criminal,Convicted - Non-violent,Trump 2
2,2025-04-04,Harlingen,1 Convicted Criminal,Convicted - Non-violent,Trump 2
3,2025-08-06,El Paso,1 Convicted Criminal,Convicted - Non-violent,Trump 2
4,2025-08-06,El Paso,1 Convicted Criminal,Convicted - Non-violent,Trump 2
...,...,...,...,...,...
375510,2017-10-10,Dallas,Convicted,Convicted - Non-violent,Trump 1
375511,2017-10-13,Dallas,No Criminal Charges,No Criminal Charges,Trump 1
375512,2017-10-07,San Diego,Convicted,Convicted - Non-violent,Trump 1
375513,2017-10-05,Los Angeles,No Criminal Charges,No Criminal Charges,Trump 1


***Creating Chart Dataset***

In [ ]:
combined_data['Apprehension Date'] = pd.to_datetime(combined_data['Apprehension Date'])

#Create date ranges to exclude the gap between 2017 and 2025
date_range_1 = pd.date_range(
    start=combined_data['Apprehension Date'].min(), 
    end='2017-10-15', 
    freq='D'
)

date_range_2 = pd.date_range(
    start='2024-10-01',
    end=combined_data['Apprehension Date'].max(), 
    freq='D'
)

#Create combinations separately for each administration period
from itertools import product

#Trump 1 combinations (up to Oct 15, 2017)
trump1_combinations = pd.DataFrame(
    list(product(
        date_range_1,
        combined_data['Apprehension AOR'].unique(),
        ['Trump 1'],
        combined_data['Type'].unique()
    )),
    columns=['Apprehension Date', 'Apprehension AOR', 'Administration', 'Type']
)

#Trump 2 combinations (from Oct 1, 2024 onwards)
trump2_combinations = pd.DataFrame(
    list(product(
        date_range_2,
        combined_data['Apprehension AOR'].unique(),
        ['Trump 2'],
        combined_data['Type'].unique()
    )),
    columns=['Apprehension Date', 'Apprehension AOR', 'Administration', 'Type']
)

#Combine both administration periods
all_combinations = pd.concat([trump1_combinations, trump2_combinations], ignore_index=True)

#Group individual AOR data
chartdf = combined_data.groupby(['Apprehension Date', 'Apprehension AOR', 'Administration', 'Type']).size().reset_index(name='Arrests')

# Merging and fill missing values with 0
chartdf = all_combinations.merge(chartdf, how='left', on=['Apprehension Date', 'Apprehension AOR', 'Administration', 'Type'])
chartdf['Arrests'] = chartdf['Arrests'].fillna(0)

#Create national aggregate
trump1_national = pd.DataFrame(
    list(product(
        date_range_1,
        ['Trump 1'],
        combined_data['Type'].unique()
    )),
    columns=['Apprehension Date', 'Administration', 'Type']
)

trump2_national = pd.DataFrame(
    list(product(
        date_range_2,
        ['Trump 2'],
        combined_data['Type'].unique()
    )),
    columns=['Apprehension Date', 'Administration', 'Type']
)

national_combinations = pd.concat([trump1_national, trump2_national], ignore_index=True)
national_aggregate = combined_data.groupby(['Apprehension Date', 'Type', 'Administration']).size().reset_index(name='Arrests')

#Merge national aggregate
national_aggregate = national_combinations.merge(national_aggregate, how='left', on=['Apprehension Date', 'Administration', 'Type'])
national_aggregate['Arrests'] = national_aggregate['Arrests'].fillna(0)
national_aggregate['Apprehension AOR'] = 'National'

#Combine individual AORs with national aggregate and sort
chartdf = pd.concat([chartdf, national_aggregate], ignore_index=True)
chartdf = chartdf.sort_values(by=['Apprehension AOR', 'Administration', 'Type', 'Apprehension Date']).reset_index(drop=True)

#Calculate rolling averages for each AOR and administration
chartdf['Week Rolling Average'] = chartdf.groupby(['Apprehension AOR', 'Administration', 'Type'])['Arrests'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
).round(2)

In [370]:
chartdf.to_csv('data/area_chart_data.csv', index=False)